# ☕🤖 RoboBarista – 3 Kameras → FAIRINO NewSpline

**Dies ist der neue Standardablauf.** Wir zeichnen eine virtuelle Kannenbewegung mit drei RGB-D-Kameras und AprilTags auf und übergeben ihre 6D-Wegpunkte danach an die offiziellen FAIRINO-Bewegungsfunktionen `NewSpline` bzw. `MoveL` mit Blend.

**Kein eigener Ruckig- oder Drehmomentregler mehr in dieser Notebook-Pipeline.**

Dieses Notebook ist ein **Offline-Trockenlauf**: Kein echter FAIRINO-Controller wird verbunden. Wir erzeugen gültige SDK-Befehle, aber keine von der Herstellerfirmware berechnete Roboterbewegung.

Die Aufnahme ist eine virtuelle, aus dem MuJoCo-Modell generierte Demonstration – keine echte menschliche Latte-Art-Aufzeichnung.


In [ ]:
# Colab: CPU genügt; die ersten virtuellen RGB-D-Render können dauern.
import os,subprocess,json
from pathlib import Path
if not Path("/content/robocasa").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Coffee-Tobi/robocasa.git",
                    "/content/robocasa"],check=True)
os.chdir("/content/robocasa")
subprocess.run(["apt-get","update","-qq"],check=True)
subprocess.run(["apt-get","install","-y","-qq","libosmesa6-dev","libgl1","ffmpeg"],check=True)
subprocess.run(["python","-m","pip","install","-q","-r","requirements.txt"],check=True)
env={**os.environ,"MUJOCO_GL":"osmesa","PYOPENGL_PLATFORM":"osmesa"}
print("RoboBarista FAIRINO pipeline bereit.")

## Schritt 1 – Mit drei RGB-D-Kameras AprilTags aufnehmen

Der virtuelle Roboter generiert eine Demonstration. **Nur die drei gerenderten Kamerabilder** werden zur 6D-Poseberechnung verwendet; der Quellweg dient nur zur nachträglichen Auswertung.

Mit `--capture-only` unterbleibt der alte IK-/Ruckig-Nachfahrversuch.

In [ ]:
subprocess.run(["python","-u","-m","robobarista.camera_joint_replay",
    "--capture-only","--seed","14","--frames","31","--fps","15",
    "--width","960","--height","720","--output","runs/fairino_camera_capture"],
    check=True,env=env)
camera=json.loads(Path("runs/fairino_camera_capture/camera_measurements.json").read_text())
print("AprilTag-Frames:",len(camera),"gültig:",sum(x["valid"] for x in camera))
print("Kameras sichtbar:",sorted({x["visible_views"] for x in camera}))
# NEU: Exportiert aus denselben AprilTag-Messungen einen portablen
# Zwischendatensatz. Die Original-Posen liegen nur separat zur Bewertung.
from robobarista.latte_dataset_bridge import export_camera_capture
export_report=export_camera_capture(
    "runs/fairino_camera_capture","runs/apriltag_dataset_export")
print("Trainingsdaten-Export:",json.dumps(export_report,indent=2))

## Schritt 2 – FAIRINO NewSpline aus diesen Kamerapunkten vorbereiten

Die Demonstration wird für unsere aktuelle Übung auf **18 mm Kannenmittelpunkt-Auslenkung**, maximal 20 mm, skaliert und zeitlich auf 20 Sekunden gestreckt. Das ist **nicht** der ursprüngliche unveränderte Bewegungsverlauf. Zusätzlich wird der Ausgießerweg mit einem **nur für das bisherige virtuelle Kannenmodell bekannten** lokalen Offset überprüft; dieser ersetzt keine echte Werkzeugkalibrierung.

Wir übergeben an `NewSpline` jeden zweiten gemessenen 6D-Punkt (16 Wegpunkte) und geben die offizielle Befehlsfolge aus. Die Identitätskalibrierung gilt **nur im Offline-Test**.

In [ ]:
subprocess.run(["python","-u","-m","robobarista.fairino_barista_pipeline",
    "--input","runs/fairino_camera_capture/camera_measurements.json",
    "--output","runs/fairino_standard","--target-mm","18",
    "--duration-s","20","--stride","2",
    "--virtual-spout-offset-m","-.073","0",".117"],check=True)
summary=json.loads(Path("runs/fairino_standard/summary.json").read_text())
print(json.dumps(summary,indent=2))

## Was ist geschafft – und was noch nicht?

- **Geschafft:** drei Kameraansichten, AprilTag-Erkennung, 6D-Aufnahme, FAIRINO-Wegpunktbefehle.
- **Nicht geschafft:** einen echten FAIRINO-Controller bzw. dessen NewSpline-Interpolation starten, reale Greiferkräfte und Flüssigkeitsphysik messen oder echte Roboterbewegungen nachweisen.
- **Für Hardware noch erforderlich:** Kamera-zu-Roboter-Kalibrierung, Greifer-/Kannengriff-Versatz, Kollisionsfreigabe und explizite Bewegungserlaubnis.

Im Ordner `runs/fairino_standard` findest du die erzeugten Dateien.

In [ ]:
from IPython.display import display
from google.colab import files
from pathlib import Path
for name in ("fairino_new_spline_commands.json","summary.json"):
    p=Path("runs/fairino_standard")/name
    print(name,":",p.stat().st_size,"Bytes")
print("Video der Kameraaufnahme:",Path("runs/fairino_camera_capture/camera_capture.mp4").exists())
print("Diese Befehle sind OFFLINE-DRY-RUN, keine reale FAIRINO-Roboterbewegung.")

## Zusätzlich: kompletter Espresso- und Milchablauf mit zwei Greifern

Dies ist unser neuer 34,5-Sekunden-Film mit elf Schritten (Siebträger, Mahlen, Tamper, Brühgruppe, Becher, Espresso, Milchdosierung, Dampfdüse, Gießen und Reinigen). Die Sensoranzeigen sind SYNTHETISCH, nicht aus echten Kontakten. Die Roboterarmbewegungen zeigen weiterhin sichere Vorpositionen.

In [ ]:
# Dies erzeugt zusätzlich den kompletten 11-Schritte-Film mit Sensor-Konzeptgrafiken.
from IPython.display import Video, display
subprocess.run(["python","-u","-m","robobarista.full_barista_sensor_video",
    "--output","runs/whole_coffee_with_FT","--frames-leg","9",
    "--capture-fps","6","--repeat","2","--width","640","--height","480"],
    check=True,env=env)
display(Video("runs/whole_coffee_with_FT/complete_barista_with_sensor_concepts.mp4",
              embed=True))
print("Hinweis: vorgeprüfte Vorpositionen; Druck-/Drehmomentwerte synthetisch.")

## Hardware-Reife und wiederverwendete Open-Source-Funktionen

Unsere Kaffee-Station verwendet die **MIT-lizenzierte RoboCasa-Kaffeemaschinenlogik** zur Kontrolle, ob der Becher unter dem Espressoauslauf steht. Unpassende STEP/STL-Downloads werden automatisch als leere CAD-Daten abgelehnt.

Die neue FAIRINO-Prüfung kann später die **8 herstellerseitigen IK-Lösungen**, die tatsächliche TCP-Pose, Gelenkgeschwindigkeit, Not-Aus- und Kraftsensorwerte auslesen. Hier werden keinerlei Hardwarebefehle gesendet. Unser reiner Softwaretest ist kein Sicherheitsnachweis.

In [ ]:
from robobarista.robocasa_coffee_adapter import CoffeeMachineState
from robobarista.fairino_sdk_preflight import hardware_readiness_gate
print("Symbolische Espresso-Station:",CoffeeMachineState().demo_brew_step((.440,.205,.882)))
gates=hardware_readiness_gate()
print("Für autonomes echtes Latte Art freigegeben:",gates["hardware_ready_for_autonomous_espresso_latte_art"])
print("Fehlende Hardwaretests:",", ".join(gates["missing_evidence"]))
print("Recherche und FAIRINO-Methode: planning/OSS_BARISTA_FR5_READINESS_AUDIT.md")